# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

tree_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ]), feature_columns),
])

scaled_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns),
])

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque si ajusto el scaler antes con todo el train, la media y la desviación ya incluyen los datos del fold de validación, y eso es data leakage: el modelo ve información de los datos con los que después lo evalúo y la accuracy de validación sale optimista. Si está dentro del pipeline, en cada fold el scaler se ajusta solo con la parte de entrenamiento de ese fold y luego se aplica al fold de validación, que es lo mismo que va a pasar con datos nuevos en producción. Además así el modelo guardado ya trae su propio preprocesamiento.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([
        ("preprocessing", clone(tree_preprocessor)),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocessing", clone(scaled_preprocessor)),
        ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocessing", clone(scaled_preprocessor)),
        ("classifier", KNeighborsClassifier()),
    ]),
}
models

{'decision_tree': Pipeline(steps=[('preprocessing',
                  ColumnTransformer(transformers=[('numeric',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='median'))]),
                                                   ['alcohol', 'malic_acid',
                                                    'ash', 'alcalinity_of_ash',
                                                    'magnesium', 'total_phenols',
                                                    'flavanoids',
                                                    'nonflavanoid_phenols',
                                                    'proanthocyanins',
                                                    'color_intensity', 'hue',
                                                    'od280/od315_of_diluted_wines',
                                                    'proline'])])),
                 ('c

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    rows.append({
        "model": name,
        "train_accuracy": scores["train_score"].mean(),
        "val_accuracy": scores["test_score"].mean(),
        "val_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
initial_comparison.round(4)

,model,train_accuracy,val_accuracy,val_std
0,logistic_regression,1.0000,0.9791,0.0277
1,knn,0.9771,0.9507,0.0353
2,decision_tree,1.0000,0.9022,0.0586


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "classifier__criterion": ["gini", "entropy"],
        "classifier__max_depth": [2, 3, 4, 5, None],
        "classifier__min_samples_split": [2, 5, 10],
    },
    "logistic_regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    "knn": {
        "classifier__n_neighbors": [3, 5, 7, 9, 11, 15],
        "classifier__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipeline in models.items():
    search = GridSearchCV(pipeline, param_grids[name], scoring="accuracy", cv=cv, n_jobs=-1, return_train_score=True)
    search.fit(X_train, y_train)
    searches[name] = search

    best = search.best_index_
    rows.append({
        "model": name,
        "cv_accuracy": search.best_score_,
        "cv_std": search.cv_results_["std_test_score"][best],
        "train_accuracy": search.cv_results_["mean_train_score"][best],
        "best_params": json.dumps(search.best_params_),
    })

tuned_comparison = pd.DataFrame(rows).sort_values("cv_accuracy", ascending=False).reset_index(drop=True)
tuned_comparison

,model,cv_accuracy,cv_std,train_accuracy,best_params
0,logistic_regression,0.979064,0.027713,1.000000,"{""classifier__C"": 1}"
1,knn,0.957882,0.040520,0.964773,"{""classifier__n_neighbors"": 7, ""classifier__we..."
2,decision_tree,0.902217,0.058617,0.998246,"{""classifier__criterion"": ""gini"", ""classifier_..."


## 6. Selección y persistencia del campeón

In [8]:
champion_name = tuned_comparison.loc[0, "model"]
champion = searches[champion_name].best_estimator_

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "selection_metric": "accuracy",
    "cv_folds": 5,
    "cv_score": float(searches[champion_name].best_score_),
    "best_params": searches[champion_name].best_params_,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(json.dumps(training_metadata, indent=2), encoding="utf-8")

tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

print("Modelo campeon:", champion_name)
print("Accuracy CV:", round(training_metadata["cv_score"], 4))
print("Parametros:", training_metadata["best_params"])

Modelo campeon: logistic_regression
Accuracy CV: 0.9791
Parametros: {'classifier__C': 1}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Escogí la **regresión logística** con C = 1 porque fue la que tuvo mejor accuracy en validación cruzada (0.979) y además la menor desviación entre folds (0.028), o sea que es la más estable. KNN con 7 vecinos quedó segundo con 0.958 y desviación 0.041, y el árbol de decisión fue el peor con 0.902 y la desviación más alta (0.059).
>
> En sobreajuste, el árbol tiene accuracy de train casi 1.0 pero 0.90 en validación, entonces está memorizando bastante. La regresión logística también llega a 1.0 en train, pero su validación está muy cerca (0.979), así que la diferencia es pequeña. KNN es el que tiene train y validación más parecidos (0.965 y 0.958).
>
> En complejidad la regresión logística es un modelo simple, rápido y con un solo hiperparámetro, y es interpretable porque cada clase tiene un coeficiente por variable. El árbol sería más fácil de explicar con reglas, pero pierde mucho desempeño. Por todo eso la regresión logística es la mejor opción.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.